# AR(3): minimized empirical width as the sample size increases

For one fixed AR(3) process, plot $T\mapsto\widehat m^{*}_{k,T}$ for $k\in\{2,3,4,5\}$, where

$$\widehat m^{*}_{k,T}=\widehat w_{k,T}(\widehat u_{k,T}),\qquad
\widehat w_{k,T}(u)=\max_{y\in\mathcal P_{k,T}} y^\top u-\min_{y\in\mathcal P_{k,T}}y^\top u,\qquad \|u\|_2=1.$$

Here $\widehat u_{k,T}$ is the best direction found by multistart projected subgradient descent. Width is piecewise linear, so this is the existing AR gradient-descent method with an active subgradient at a kink. The asterisk denotes the numerical optimizer's result; it does not certify a global minimum.

Use the simple, stationary coefficient vector $\phi=(0.6,-0.3,0.2)$ and innovations $\varepsilon_t\sim\operatorname{Unif}[-1,1]$:

$$X_t=0.6X_{t-1}-0.3X_{t-2}+0.2X_{t-3}+\varepsilon_t.$$

After a burn-in, simulate **one long path** and use nested prefixes of length $T$. Following the common-endpoint convention in the AR chapter, set $K=5$ and use $\mathcal P_{k,T}=\{(X_t,\ldots,X_{t-k+1}):t=K,\ldots,T\}$. Thus each dimension has exactly $T-K+1$ points, and $T$ is the number of retained time-series observations, not the number of optimizer steps. No sample-dependent rescaling is applied.

In [ ]:
import sys                                                                                          ### INFRASTRUCTURE
from pathlib import Path                                                                            ### INFRASTRUCTURE
from time import perf_counter
import json                                                                                         ### INFRASTRUCTURE

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt                                                                     ### PLOTTING
from IPython.display import display                                                                 ### REPORTING
from threadpoolctl import threadpool_limits                                                         ### INFRASTRUCTURE

AR_CODE_DIR = next(                                                                                 ### INFRASTRUCTURE
    (                                                                                               ### INFRASTRUCTURE
        candidate.resolve()                                                                         ### INFRASTRUCTURE
        for root in (Path.cwd(), *Path.cwd().parents)                                               ### INFRASTRUCTURE
        for candidate in (root / "Code" / "Code_AR_processes", root)                                ### INFRASTRUCTURE
        if (candidate / "ar_tools/functions.py").is_file()                                          ### INFRASTRUCTURE
    ),                                                                                              ### INFRASTRUCTURE
    None,                                                                                           ### INFRASTRUCTURE
)                                                                                                   ### INFRASTRUCTURE
if AR_CODE_DIR is None:                                                                             ### INFRASTRUCTURE
    raise FileNotFoundError("Could not locate Code/Code_AR_processes.")                             ### INFRASTRUCTURE
PROJECT_ROOT = AR_CODE_DIR.parents[1]                                                               ### INFRASTRUCTURE
if str(PROJECT_ROOT) not in sys.path:                                                               ### INFRASTRUCTURE
    sys.path.insert(0, str(PROJECT_ROOT))                                                           ### INFRASTRUCTURE

from Code.Code_AR_processes.ar_tools.descent import (
    batched_geometric_width_curves,
)
from Code.Code_AR_processes.ar_tools.functions import (
    is_stationary,
    simulate_uniform_ar,
    sliding_window_point_cloud,
    theoretical_ar_width_in_direction,
)

EXPERIMENT_NAME = "ar3_minimum_width_vs_T"                                                          ### INFRASTRUCTURE
CSV_DIR = AR_CODE_DIR / "outputs" / "csv" / EXPERIMENT_NAME / "absolute_steps" / "mae"                                         ### INFRASTRUCTURE
PDF_DIR = AR_CODE_DIR / "outputs" / "pdfs" / EXPERIMENT_NAME / "absolute_steps" / "mae"                                        ### INFRASTRUCTURE
IMAGE_DIR = AR_CODE_DIR / "outputs" / "images" / EXPERIMENT_NAME / "absolute_steps" / "mae"                                    ### INFRASTRUCTURE
for directory in (CSV_DIR, PDF_DIR, IMAGE_DIR):                                                     ### INFRASTRUCTURE
    directory.mkdir(parents=True, exist_ok=True)                                                    ### INFRASTRUCTURE

plt.rcParams.update({"font.size": 11, "axes.spines.top": False,                                     ### PLOTTING
                     "axes.spines.right": False, "pdf.fonttype": 42})                               ### PLOTTING
np.set_printoptions(precision=6, suppress=True)                                                     ### REPORTING

from Code.Code_AR_processes.ar_tools.coefficient_errors import coefficient_mae


## Experiment settings

Edit this cell to change the coefficients, sample-size grid, or descent budget. The same deterministic random starts are reused for a given $k$ at every $T$. All starts are optimized, and the smallest attained width is retained. The true AR direction is not used to initialize the estimator.

In [ ]:
phi = np.array([0.6, -0.3, 0.2], dtype=float)
k_values = np.array([2, 3, 4, 5], dtype=int)
T_values = np.array([100, 200, 500, 1_000, 2_000, 5_000,
                     10_000, 20_000, 50_000], dtype=int)
K = int(k_values.max())
burnin = 2_000
innovations_half_width = 1.0
process_seed = 20260916
optimizer_seed = 20260917
number_of_starts = 12
optimization_iterations = 3_000
initial_step = 0.2
geometric_decay = 0.995
population_history = 1_000

stationary, companion_eigenvalues = is_stationary(phi)
if len(phi) != 3 or phi[-1] == 0 or not stationary:
    raise ValueError("Choose a stationary, genuine AR(3) coefficient vector.")
if (T_values.ndim != 1 or len(T_values) < 2
        or np.any(np.diff(T_values) <= 0) or T_values[0] <= K):
    raise ValueError("T_values must be strictly increasing and exceed K.")
if (k_values.ndim != 1 or len(k_values) == 0
        or len(np.unique(k_values)) != len(k_values)
        or np.any(k_values < 2)):
    raise ValueError("k_values must contain distinct embedding dimensions >= 2.")
if number_of_starts < 1 or optimization_iterations < 1:
    raise ValueError("Use positive numbers of starts and iterations.")
if initial_step <= 0 or not 0 < geometric_decay < 1:
    raise ValueError("Require initial_step > 0 and 0 < geometric_decay < 1.")

ar3_series = simulate_uniform_ar(
    phi, sample_size=int(T_values[-1]), burnin=burnin,
    half_width=innovations_half_width, rng=process_seed,
)
population_plateau = (
    2.0 * innovations_half_width / np.sqrt(1.0 + phi @ phi)
)

print(f"Coefficients: {phi}")                                                                       ### REPORTING
print(f"Companion spectral radius: {max(abs(companion_eigenvalues)):.6f} < 1")                      ### REPORTING
print(f"Sample sizes T: {T_values}")                                                                ### REPORTING
print(f"Dimensions k: {k_values}; common endpoint K = {K}")                                         ### REPORTING
print(f"Starts: {number_of_starts}; iterations per start: {optimization_iterations}")               ### REPORTING
print(f"Population reference for k >= 4: {population_plateau:.8f}")                                 ### REPORTING

## Minimize the width for every $(k,T)$

The optimizer uses the shared `batched_geometric_width_curves` implementation. For each dimension it runs all starts together with the sphere update

$$g_n=Y_{\max} - Y_{\min},\qquad g_n^{\mathrm{tan}}=g_n-(u_n^\top g_n)u_n,\qquad
u_{n+1}=\frac{u_n-\eta_n g_n^{\mathrm{tan}}}{\|u_n-\eta_n g_n^{\mathrm{tan}}\|_2}.$$

Use absolute steps $\eta_n=0.2(0.995)^n$. The settings cell controls the initial step and decay. The reported width is recomputed from the saved unit direction to check that the optimization result and point cloud agree.


In [ ]:
experiment_rows = []
estimated_directions = {}
experiment_start = perf_counter()

# Small-dimensional matrix products are faster with one BLAS thread.                                ### INFRASTRUCTURE
with threadpool_limits(limits=1):                                                                   ### INFRASTRUCTURE
    for T in T_values:
        T = int(T)
        for k in k_values:
            k = int(k)
            # In zero-based indexing, this slice makes all cloud endpoints
            # run from K - 1 through T - 1, with newest-first coordinates.
            series_for_k = ar3_series[K - k:T]
            cloud = sliding_window_point_cloud(series_for_k, k)
            if cloud.shape != (T - K + 1, k):
                raise RuntimeError("The common-endpoint embedding is inconsistent.")
            start_time = perf_counter()
            single_widths, best_widths, _, directions = (
                batched_geometric_width_curves(
                    series_for_k, k, seed=optimizer_seed, replication=0,
                    starts=number_of_starts, iterations=optimization_iterations,
                    initial_step=initial_step, geometric_decay=geometric_decay,
                    return_directions=True,
                )
            )
            optimization_seconds = perf_counter() - start_time
            direction = directions[k - 1]
            projections = cloud @ direction
            empirical_width = float(np.ptp(projections))
            if not np.isclose(np.linalg.norm(direction), 1.0, atol=1e-10):
                raise RuntimeError("The estimated direction is not a unit vector.")
            if not np.isclose(empirical_width, best_widths[k - 1], atol=1e-10):
                raise RuntimeError("The saved direction does not reproduce the width.")
            population_width_at_estimate = theoretical_ar_width_in_direction(
                phi, direction, half_width=innovations_half_width,
                history=population_history,
            )
            estimated_directions[(k, T)] = direction.copy()
            row = {
                "T": T, "k": k, "point_count": len(cloud),
                "minimum_width": empirical_width,
                "first_random_start_width": float(single_widths[k - 1]),
                "restart_improvement": float(single_widths[k - 1] - empirical_width),
                "population_width_at_estimated_direction": population_width_at_estimate,
                "initial_step": initial_step,
                "optimization_seconds": optimization_seconds,
            }
            for coordinate in range(K):
                row[f"u_{coordinate}"] = (
                    float(direction[coordinate]) if coordinate < k else np.nan
                )
            experiment_rows.append(row)
        latest = experiment_rows[-len(k_values):]
        print(f"T = {T:>6,}: " + ", ".join(                                                         ### REPORTING
            f"k={row['k']}: {row['minimum_width']:.6f}" for row in latest                           ### REPORTING
        ), flush=True)                                                                              ### REPORTING

width_results = pd.DataFrame(experiment_rows).sort_values(["T", "k"])
width_table = width_results.pivot(index="T", columns="k", values="minimum_width")
width_results.to_csv(CSV_DIR / "width_results.csv", index=False)                                    ### INFRASTRUCTURE
width_table.to_csv(CSV_DIR / "width_vs_T.csv")                                                      ### INFRASTRUCTURE
configuration = {                                                                                   ### INFRASTRUCTURE
    "phi": phi.tolist(), "k_values": k_values.tolist(), "T_values": T_values.tolist(),              ### INFRASTRUCTURE
    "K": K, "burnin": burnin, "innovations_half_width": innovations_half_width,                     ### INFRASTRUCTURE
    "process_seed": process_seed, "optimizer_seed": optimizer_seed,                                 ### INFRASTRUCTURE
    "number_of_starts": number_of_starts,                                                           ### INFRASTRUCTURE
    "optimization_iterations": optimization_iterations,                                             ### INFRASTRUCTURE
    "initial_step": initial_step, "geometric_decay": geometric_decay,                   ### INFRASTRUCTURE
    "population_history": population_history,                                                       ### INFRASTRUCTURE
    "optimizer": "batched_geometric_width_curves",                                                  ### INFRASTRUCTURE
    "embedding": "newest first, common endpoints t = K,...,T",                                      ### INFRASTRUCTURE
}                                                                                                   ### INFRASTRUCTURE
(CSV_DIR / "configuration.json").write_text(json.dumps(configuration, indent=2) + "\n")             ### INFRASTRUCTURE
print(f"Total elapsed time: {perf_counter() - experiment_start:.1f} seconds")                       ### REPORTING
display(width_table.round(6))                                                                       ### REPORTING

## Plot $T$ versus $\widehat m^{*}_{k,T}$

Both panels show the same four raw curves. The logarithmic $T$ axis makes changes at smaller sample sizes visible; the linear axis shows the large-$T$ behavior. The horizontal reference is

$$m_4=m_5=\frac{2}{\sqrt{1+0.6^2+(-0.3)^2+0.2^2}}.$$

This reference concerns $k\ge p+1=4$; it is not the population minimum for $k=2,3$. It is calculated from the configured coefficients and innovation support, and is not used by the estimator.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.8), constrained_layout=True)                        ### PLOTTING
colors = {int(k): plt.get_cmap("tab10")(index)                                                      ### PLOTTING
          for index, k in enumerate(k_values)}                                                      ### PLOTTING
markers = ["o", "s", "^", "D", "v", "P", "X"]                                                       ### PLOTTING

for ax in axes:                                                                                     ### PLOTTING
    for index, k in enumerate(k_values):                                                            ### PLOTTING
        ax.plot(                                                                                    ### PLOTTING
            width_table.index, width_table[int(k)],                                                 ### PLOTTING
            color=colors[int(k)], marker=markers[index % len(markers)],                             ### PLOTTING
            linewidth=1.8, markersize=5, label=fr"$k={k}$",                                         ### PLOTTING
        )                                                                                           ### PLOTTING
    ax.axhline(                                                                                     ### PLOTTING
        population_plateau, color="0.25", linestyle="--", linewidth=1.3,                            ### PLOTTING
        label=r"Population minimum for $k\geq 4$",                                                  ### PLOTTING
    )                                                                                               ### PLOTTING
    ax.set_xlabel(r"Sample size $T$")                                                               ### PLOTTING
    ax.set_ylabel(r"Minimized empirical width $\widehat{m}^{*}_{k,T}$")                             ### PLOTTING
    ax.grid(alpha=0.2)                                                                              ### PLOTTING
    ax.legend(fontsize=9, loc="best")                                                               ### PLOTTING
axes[0].set_title("Linear sample-size axis")                                                        ### PLOTTING
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)                                  ### PLOTTING
axes[1].set_title("Logarithmic sample-size axis")                                                   ### PLOTTING
axes[1].set_xscale("log")                                                                           ### PLOTTING
fig.suptitle(                                                                                       ### PLOTTING
    fr"AR(3), $\phi=({phi[0]:g},{phi[1]:g},{phi[2]:g})$, "                                          ### PLOTTING
    r"$\varepsilon_t\sim\mathrm{Unif}[-1,1]$"                                                       ### PLOTTING
)                                                                                                   ### PLOTTING
figure_stem = "ar3_minimum_width_vs_T"                                                              ### PLOTTING
fig.savefig(PDF_DIR / f"{figure_stem}.pdf", bbox_inches="tight")                                    ### PLOTTING
fig.savefig(IMAGE_DIR / f"{figure_stem}.png", dpi=180, bbox_inches="tight")                         ### PLOTTING
plt.show()                                                                                          ### PLOTTING
plt.close(fig)                                                                                      ### PLOTTING
print(f"PDF: {PDF_DIR / (figure_stem + '.pdf')}")                                                   ### REPORTING
print(f"CSV: {CSV_DIR / 'width_results.csv'}")                                                      ### REPORTING

## Numerical diagnostics and interpretation

For fixed $k$, the exact empirical minimum is nondecreasing along nested point clouds. Independent finite-budget descent runs can violate that property if they find different local minima; the table and plot retain the raw estimates. The diagnostic below reports any downward steps and the benefit of extra starts rather than imposing monotonicity.

The population width evaluated at the estimated direction helps interpret the remaining gap: its difference from the empirical width measures how incompletely the sampled points fill the support in that direction. For $k=4,5$, compare it with the population reference to inspect how close the direction is to a population minimizer. A finite iteration budget does not establish convergence as $T$ tends to infinity. This is one sample-path experiment, not a Monte Carlo mean.

In [ ]:
downward_steps = []                                                                                 ### REPORTING
for k in k_values:                                                                                  ### REPORTING
    values = width_table[int(k)].to_numpy()                                                         ### REPORTING
    for index in np.flatnonzero(np.diff(values) < -1e-7):                                           ### REPORTING
        downward_steps.append({                                                                     ### REPORTING
            "k": int(k), "T_from": int(T_values[index]),                                            ### REPORTING
            "T_to": int(T_values[index + 1]),                                                       ### REPORTING
            "width_change": float(values[index + 1] - values[index]),                               ### REPORTING
        })                                                                                          ### REPORTING
if downward_steps:                                                                                  ### REPORTING
    print("Downward steps in the raw optimizer estimates:")                                         ### REPORTING
    display(pd.DataFrame(downward_steps))                                                           ### REPORTING
else:                                                                                               ### REPORTING
    print("No downward steps larger than 1e-7 in the raw width curves.")                            ### REPORTING

final_diagnostics = width_results.loc[                                                              ### REPORTING
    width_results["T"] == T_values[-1],                                                             ### REPORTING
    ["k", "minimum_width", "first_random_start_width", "restart_improvement",                       ### REPORTING
     "population_width_at_estimated_direction"],                                                    ### REPORTING
].set_index("k")                                                                                    ### REPORTING
print(f"Diagnostics at T = {T_values[-1]:,}:")                                                      ### REPORTING
display(final_diagnostics.round(6))                                                                 ### REPORTING
print(f"Population reference for k >= 4: {population_plateau:.8f}")                                 ### REPORTING

## Convergence speed: $T$ versus $|m_k-\widehat m_{k,T}|$

Use the optimized empirical widths above and compute the **absolute error relative to the population minimum** separately for every $k$:

$$e_k(T)=\left|m_k-\widehat m^{*}_{k,T}\right|.$$

For $k=4,5$, use the existing population formula. For $k=2,3$, numerically minimize the population support width of the finite-history AR zonotope. In two dimensions its minimum occurs at a generator normal; in three dimensions it occurs at a normal to a pair of independent generators. Enumerating these directions gives the finite-zonotope minimum, rather than evaluating population width at the empirical direction. Check it against twice the impulse-response history; this measures truncation sensitivity, not a rigorous bound on the infinite tail.

The second panel uses logarithmic axes: a power law $e_k(T)\approx CT^{-\alpha}$ appears as a line with slope $-\alpha$. The gray $T^{-1/2}$ and $T^{-1}$ lines are visual comparisons with an arbitrary common scale. The fitted exponents use the last five available sample sizes and describe this single path only. The plotted errors include sampling and finite-budget optimization effects; small errors can also result from cancellation between those effects.

In [ ]:
from Code.Code_AR_processes.ar_tools.functions import ar_window_zonotope_generators


from Code.Code_AR_processes.ar_tools.population_widths import population_minimum_low_dimension


population_reference_history = max(1_000, int(population_history))
population_reference_rows = []
with threadpool_limits(limits=1):                                                                   ### INFRASTRUCTURE
    for k in k_values:
        k = int(k)
        if k < len(phi) + 1:
            minimum, _ = population_minimum_low_dimension(
                phi, k, population_reference_history, innovations_half_width,
            )
            longer_history_minimum, _ = population_minimum_low_dimension(
                phi, k, 2 * population_reference_history, innovations_half_width,
            )
            history_change = abs(longer_history_minimum - minimum)
            if history_change > 1e-9:
                raise RuntimeError(
                    f"Population reference for k={k} is sensitive to truncation; "
                    "increase population_reference_history."
                )
            minimum = longer_history_minimum
            method = "finite-zonotope normal enumeration"
        else:
            minimum = population_plateau
            history_change = 0.0
            method = "population formula"
        population_reference_rows.append({
            "k": k, "population_minimum": float(minimum),
            "history_doubling_change": float(history_change), "method": method,
        })

population_references = pd.DataFrame(population_reference_rows).set_index("k")
population_minima = population_references["population_minimum"]
convergence_results = width_results[["T", "k", "minimum_width"]].copy()
convergence_results["population_minimum"] = convergence_results["k"].map(population_minima)
convergence_results["absolute_error"] = np.abs(
    convergence_results["population_minimum"] - convergence_results["minimum_width"]
)
error_table = convergence_results.pivot(index="T", columns="k", values="absolute_error")
population_references.to_csv(CSV_DIR / "population_minimum_references.csv")                         ### INFRASTRUCTURE
convergence_results.to_csv(CSV_DIR / "convergence_errors.csv", index=False)                         ### INFRASTRUCTURE
error_table.to_csv(CSV_DIR / "absolute_error_vs_T.csv")                                             ### INFRASTRUCTURE
print("Population reference for each embedding dimension:")                                         ### REPORTING
display(population_references)                                                                      ### REPORTING
print("Absolute convergence errors:")                                                               ### REPORTING
display(error_table.round(8))                                                                       ### REPORTING

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.8), constrained_layout=True)                        ### PLOTTING
convergence_colors = {                                                                              ### PLOTTING
    int(k): plt.get_cmap("tab10")(index) for index, k in enumerate(k_values)                        ### PLOTTING
}                                                                                                   ### PLOTTING
convergence_markers = ["o", "s", "^", "D", "v", "P", "X"]                                           ### PLOTTING

for index, k in enumerate(k_values):                                                                ### PLOTTING
    errors = error_table[int(k)].to_numpy()                                                         ### PLOTTING
    sample_sizes = error_table.index.to_numpy(dtype=float)                                          ### PLOTTING
    label = fr"$k={k}$"                                                                             ### PLOTTING
    style = dict(                                                                                   ### PLOTTING
        color=convergence_colors[int(k)],                                                           ### PLOTTING
        marker=convergence_markers[index % len(convergence_markers)],                               ### PLOTTING
        linewidth=1.8, markersize=5, label=label,                                                   ### PLOTTING
    )                                                                                               ### PLOTTING
    axes[0].plot(sample_sizes, errors, **style)                                                     ### PLOTTING
    # A zero cannot be displayed on logarithmic axes; retain it in the CSV                          ### PLOTTING
    # and linear panel rather than replacing it with a fabricated value.                            ### PLOTTING
    positive = errors > 0                                                                           ### PLOTTING
    axes[1].loglog(sample_sizes[positive], errors[positive], **style)                               ### PLOTTING
    if np.any(~positive):                                                                           ### REPORTING
        print(f"k={k}: {np.count_nonzero(~positive)} zero errors omitted from log-log panel.")      ### REPORTING

first_errors = error_table.iloc[0].to_numpy()                                                       ### PLOTTING
positive_first_errors = first_errors[first_errors > 0]                                              ### PLOTTING
if len(positive_first_errors):                                                                      ### PLOTTING
    guide_scale = float(np.median(positive_first_errors))                                           ### PLOTTING
    guide_T = np.geomspace(error_table.index.min(), error_table.index.max(), 200)                   ### PLOTTING
    for exponent, linestyle, label in (                                                             ### PLOTTING
        (0.5, ":", r"$T^{-1/2}$ comparison"),                                                       ### PLOTTING
        (1.0, "--", r"$T^{-1}$ comparison"),                                                        ### PLOTTING
    ):                                                                                              ### PLOTTING
        axes[1].loglog(                                                                             ### PLOTTING
            guide_T, guide_scale * (guide_T / guide_T[0]) ** (-exponent),                           ### PLOTTING
            color="0.5", linestyle=linestyle, linewidth=1.2, label=label,                           ### PLOTTING
        )                                                                                           ### PLOTTING

for ax in axes:                                                                                     ### PLOTTING
    ax.set_xlabel(r"Sample size $T$")                                                               ### PLOTTING
    ax.set_ylabel(r"Absolute error $|m_k-\widehat{m}^{*}_{k,T}|$")                                  ### PLOTTING
    ax.grid(which="both", alpha=0.2)                                                                ### PLOTTING
    ax.legend(fontsize=9, loc="best")                                                               ### PLOTTING
axes[0].set_title("Absolute error versus sample size")                                              ### PLOTTING
axes[0].set_ylim(bottom=0)                                                                          ### PLOTTING
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)                                  ### PLOTTING
axes[1].set_title("Log-log view of convergence speed")                                              ### PLOTTING
fig.suptitle("AR(3): convergence of optimized empirical minimum widths")                            ### PLOTTING
convergence_figure_stem = "ar3_minimum_width_convergence_error"                                     ### PLOTTING
fig.savefig(PDF_DIR / f"{convergence_figure_stem}.pdf", bbox_inches="tight")                        ### PLOTTING
fig.savefig(IMAGE_DIR / f"{convergence_figure_stem}.png", dpi=180, bbox_inches="tight")             ### PLOTTING
plt.show()                                                                                          ### PLOTTING
plt.close(fig)                                                                                      ### PLOTTING

# Descriptive least-squares slope on a configurable large-T window.
convergence_fit_points = 5
fit_rows = []
for k in k_values:
    tail_errors = error_table[int(k)].iloc[-convergence_fit_points:]
    tail_errors = tail_errors[tail_errors > 0]
    if len(tail_errors) < 2:
        continue
    log_T = np.log(tail_errors.index.to_numpy(dtype=float))
    log_error = np.log(tail_errors.to_numpy())
    slope, intercept = np.polyfit(log_T, log_error, deg=1)
    fit_rows.append({
        "k": int(k), "fit_T_min": int(tail_errors.index.min()),
        "fit_T_max": int(tail_errors.index.max()), "fit_points": len(tail_errors),
        "log_log_slope": float(slope), "descriptive_alpha": float(-slope),
    })
convergence_slopes = pd.DataFrame(fit_rows)
convergence_slopes.to_csv(CSV_DIR / "descriptive_convergence_slopes.csv", index=False)              ### INFRASTRUCTURE
print("Descriptive power-law fit: error approximately C * T**(-alpha)")                             ### REPORTING
print("Single-path slopes include finite-budget optimization effects.")                             ### REPORTING
display(convergence_slopes.round(4))                                                                ### REPORTING
print(f"PDF: {PDF_DIR / (convergence_figure_stem + '.pdf')}")                                       ### REPORTING

## Coefficient MAE versus sample size

Recover the minimum-width coefficients from the k=p+1 direction, and fit Yule–Walker, Gaussian conditional QML, and Horváth L-infinity to the same nested sample prefixes. Compare every method using

$$\operatorname{MAE}(T)=\frac1p\sum_{j=1}^p|\widehat\phi_{j,T}-\phi_j|.$$

The two panels show this coefficient MAE against sample size on linear and logarithmic axes. The estimator fitting objectives are retained; MAE is the common measure of coefficient accuracy. Estimated coefficients and the QML and LP fit diagnostics are saved alongside each score. [Shared accuracy definition](../ar_tools/coefficient_errors.py), `coefficient_mae`; [estimator implementations](../ar_tools/classical_selectors.py).

In [ ]:
from Code.Code_AR_processes.ar_tools.classical_selectors import (
    yule_walker_coefficients, conditional_gaussian_qml, horvath_linf_coefficients,
)

coefficient_embedding_dimension = len(phi) + 1
coefficient_width_results = width_results.loc[
    width_results["k"] == coefficient_embedding_dimension
].sort_values("T")
if coefficient_width_results.empty:
    raise ValueError("The coefficient comparison requires the k=p+1 embedding.")
coefficient_rows = []
with threadpool_limits(limits=1):
    for record in coefficient_width_results.to_dict("records"):
        T = int(record["T"])
        direction = np.array([record[f"u_{j}"] for j in range(len(phi) + 1)])
        minimum_width_phi = np.full(len(phi), np.nan)
        if np.all(np.isfinite(direction)) and abs(direction[0]) >= 1e-10:
            minimum_width_phi = -direction[1:] / direction[0]
        qml_phi, variance, log_likelihood = conditional_gaussian_qml(ar3_series[:T], len(phi))
        lp_phi, radius, gap = horvath_linf_coefficients(ar3_series[:T], len(phi))
        estimates = {
            "Gradient descent": minimum_width_phi,
            "Yule-Walker": yule_walker_coefficients(ar3_series[:T], len(phi)),
            "Conditional QML": qml_phi,
            "Horvath L-infinity LP": lp_phi,
        }
        for method, estimate in estimates.items():
            row = {"T": T, "method": method, "coefficient_mae": coefficient_mae(estimate, phi)}
            row.update({f"estimated_phi_{j+1}": float(value) for j, value in enumerate(estimate)})
            if method == "Conditional QML":
                row.update(residual_variance=variance, quasi_log_likelihood=log_likelihood)
            elif method == "Horvath L-infinity LP":
                row.update(linf_radius=radius, linf_duality_gap=gap)
            coefficient_rows.append(row)
coefficient_mae_results = pd.DataFrame(coefficient_rows)
coefficient_mae_results.to_csv(CSV_DIR / "coefficient_mae_vs_T.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.8), constrained_layout=True)
for ax in axes:
    for method, values in coefficient_mae_results.groupby("method", sort=False):
        ax.plot(values["T"], values["coefficient_mae"], marker="o", label=method)
    ax.set(xlabel="Sample size T", ylabel="Coefficient MAE", ylim=(0, None))
    ax.grid(alpha=.2)
    ax.legend(fontsize=9)
axes[0].set_title("Linear sample-size axis")
axes[1].set(title="Logarithmic sample-size axis", xscale="log")
fig.suptitle("AR(3): coefficient MAE from four estimators")
fig.savefig(PDF_DIR / "ar3_coefficient_mae_vs_T.pdf", bbox_inches="tight")
fig.savefig(IMAGE_DIR / "ar3_coefficient_mae_vs_T.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)
display(coefficient_mae_results.round(6))

## Repeated coefficient MAE experiment

Compare minimum-width gradient descent, Yule–Walker, Gaussian conditional QML, and Horváth L-infinity over R independent AR(3) paths, holding the coefficients and innovation distribution fixed. Every method fits the same nested sample prefixes. The known coefficients enter accuracy scoring after fitting.

$$\operatorname{MAE}_a(T)=\frac{1}{Rp}\sum_{r=1}^{R}\sum_{j=1}^{p}|\widehat\phi^{(r)}_{a,j,T}-\phi_j|.$$

Report the mean, median and 90th percentile of per-fit coefficient MAE, per-coefficient mean absolute errors, and nonstationary-estimate counts. Finite large errors remain in every unconditional mean. A failed fit is recorded as NaN and makes the corresponding unconditional MAE unavailable.

The shaded bands use the same bootstrap replication indices for every method and sample size. They summarize Monte Carlo uncertainty in mean MAE. The paired table compares gradient descent minus comparator MAE; a positive difference favors the comparator. The cache is identified by the MAE convention, configuration and estimator source. [Accuracy implementation](../ar_tools/coefficient_errors.py), `coefficient_mae`.

In [ ]:
import hashlib                                                                                      ### INFRASTRUCTURE
import inspect                                                                                      ### INFRASTRUCTURE
import os                                                                                           ### INFRASTRUCTURE
from joblib import Parallel, delayed, parallel_config                                               ### INFRASTRUCTURE

mae_replication_count = 100
mae_T_values = T_values.copy()
mae_phi = phi.copy()
mae_burnin = burnin
mae_half_width = innovations_half_width
mae_k = len(mae_phi) + 1
mae_K = K
mae_starts = number_of_starts
mae_iterations = optimization_iterations
mae_initial_step = initial_step
mae_geometric_decay = geometric_decay
mae_process_seed = 20260923
mae_optimizer_seed = 20260924
mae_bootstrap_seed = 20260925
mae_bootstrap_count = 2_000
mae_jobs = min(4, os.cpu_count() or 1)                                                             ### INFRASTRUCTURE
mae_resume = True                                                                                  ### INFRASTRUCTURE
mae_use_convex_hull = True

if mae_replication_count < 2 or mae_bootstrap_count < 2:
    raise ValueError("Use at least two replications and bootstrap samples.")
if (mae_T_values.ndim != 1 or len(mae_T_values) < 2
        or np.any(np.diff(mae_T_values) <= 0)
        or mae_T_values[0] <= max(mae_K, 2 * len(mae_phi))):
    raise ValueError("Use increasing sample sizes large enough for all fits.")
if not is_stationary(mae_phi)[0] or mae_K < mae_k:
    raise ValueError("Use stationary coefficients and K >= p+1.")
if mae_starts < 1 or mae_iterations < 1 or mae_jobs < 1:
    raise ValueError("Use positive optimizer budgets and worker counts.")
if mae_initial_step <= 0 or not 0 < mae_geometric_decay < 1:
    raise ValueError("Invalid descent step-size settings.")

MAE_METHODS = ["Gradient descent", "Yule-Walker", "Conditional QML", "Horvath L-infinity LP"]
MAE_DIR = CSV_DIR / "repeated_mae"                                                                ### INFRASTRUCTURE
MAE_DIR.mkdir(parents=True, exist_ok=True)                                                         ### INFRASTRUCTURE
# Include the shared optimizer/simulator and the shared estimator and experiment definitions.                       ### INFRASTRUCTURE
mae_estimator_source = (                                                                           ### INFRASTRUCTURE
    (AR_CODE_DIR / "ar_tools/functions.py").read_bytes()                                            ### INFRASTRUCTURE
    + (AR_CODE_DIR / "ar_tools/descent.py").read_bytes()
    + (AR_CODE_DIR / "ar_tools/coefficient_errors.py").read_bytes()
    + (AR_CODE_DIR / "ar_tools/experiments.py").read_bytes()                                            ### INFRASTRUCTURE
    + inspect.getsource(yule_walker_coefficients).encode()                                          ### INFRASTRUCTURE
    + inspect.getsource(conditional_gaussian_qml).encode()                                          ### INFRASTRUCTURE
    + inspect.getsource(horvath_linf_coefficients).encode()                                         ### INFRASTRUCTURE
)                                                                                                   ### INFRASTRUCTURE
mae_configuration = {                                                                              ### INFRASTRUCTURE
    "experiment_version": 4,
    "coefficient_error_metric": "mae_per_coefficient_v1",                                                                        ### INFRASTRUCTURE
    "use_convex_hull": mae_use_convex_hull,                                                        ### INFRASTRUCTURE
    "replication_count": mae_replication_count,                                                    ### INFRASTRUCTURE
    "phi": mae_phi.tolist(), "T_values": mae_T_values.tolist(),                                   ### INFRASTRUCTURE
    "burnin": mae_burnin, "half_width": mae_half_width,                                           ### INFRASTRUCTURE
    "k": mae_k, "K": mae_K, "starts": mae_starts,                                                ### INFRASTRUCTURE
    "iterations": mae_iterations, "initial_step": mae_initial_step,                   ### INFRASTRUCTURE
    "geometric_decay": mae_geometric_decay,                                                        ### INFRASTRUCTURE
    "process_seed": mae_process_seed, "optimizer_seed": mae_optimizer_seed,                       ### INFRASTRUCTURE
    "bootstrap_seed": mae_bootstrap_seed, "bootstrap_count": mae_bootstrap_count,                 ### INFRASTRUCTURE
    "estimator_source_sha256": hashlib.sha256(mae_estimator_source).hexdigest(),                   ### INFRASTRUCTURE
    "invalid_u_0_threshold": 1e-10,                                                                 ### INFRASTRUCTURE
    "methods": MAE_METHODS,                                                                        ### INFRASTRUCTURE
}                                                                                                   ### INFRASTRUCTURE
mae_configuration_text = json.dumps(mae_configuration, sort_keys=True, indent=2)                  ### INFRASTRUCTURE
mae_configuration_id = hashlib.sha256(mae_configuration_text.encode()).hexdigest()[:16]           ### INFRASTRUCTURE
MAE_CACHE_DIR = MAE_DIR / "replications" / mae_configuration_id                                  ### INFRASTRUCTURE
MAE_CACHE_DIR.mkdir(parents=True, exist_ok=True)                                                   ### INFRASTRUCTURE
(MAE_CACHE_DIR / "configuration.json").write_text(mae_configuration_text + "\n")                  ### INFRASTRUCTURE
print(f"Replications: {mae_replication_count}; sample sizes: {mae_T_values}")                     ### REPORTING
print(f"Fixed coefficients: {mae_phi}; workers: {mae_jobs}")                                      ### REPORTING
print(f"Descent: {mae_starts} starts, {mae_iterations} iterations per start")                     ### REPORTING
print(f"Configuration ID: {mae_configuration_id}")                                                 ### REPORTING

In [ ]:
from Code.Code_AR_processes.ar_tools.experiments import run_mae_replication


mae_run_start = perf_counter()
mae_replication_frames = []
# Each worker fits one whole replication; NumPy uses one BLAS thread per worker.                    ### INFRASTRUCTURE
with parallel_config(backend="loky", inner_max_num_threads=1):                                      ### INFRASTRUCTURE
    completed_replications = Parallel(n_jobs=mae_jobs, return_as="generator_unordered")(           ### INFRASTRUCTURE
        delayed(run_mae_replication)(                                                              ### INFRASTRUCTURE
            replication, mae_configuration, str(MAE_CACHE_DIR), mae_resume,                      ### INFRASTRUCTURE
        )                                                                                           ### INFRASTRUCTURE
        for replication in range(1, mae_replication_count + 1)                                     ### INFRASTRUCTURE
    )                                                                                               ### INFRASTRUCTURE
    for count, result in enumerate(completed_replications, start=1):
        mae_replication_frames.append(result)
        if count == 1 or count % 10 == 0 or count == mae_replication_count:                        ### REPORTING
            checkpoint = pd.concat(mae_replication_frames, ignore_index=True)                      ### INFRASTRUCTURE
            checkpoint.to_csv(MAE_DIR / "replication_results_partial.csv", index=False)            ### INFRASTRUCTURE
            print(f"Completed {count}/{mae_replication_count} replications; "                      ### REPORTING
                  f"elapsed {(perf_counter() - mae_run_start) / 60:.1f} minutes", flush=True)      ### REPORTING

mae_replication_results = pd.concat(mae_replication_frames, ignore_index=True).sort_values(
    ["replication", "T", "method"]
).reset_index(drop=True)
expected_count = mae_replication_count * len(mae_T_values) * len(MAE_METHODS)
if (len(mae_replication_results) != expected_count
        or mae_replication_results.duplicated(["replication", "T", "method"]).any()):
    raise RuntimeError("The paired Monte Carlo results are incomplete or duplicated.")
mae_replication_results.to_csv(MAE_DIR / "replication_results.csv", index=False)                  ### INFRASTRUCTURE
(MAE_DIR / "configuration.json").write_text(mae_configuration_text + "\n")                        ### INFRASTRUCTURE
mae_failures = mae_replication_results.loc[mae_replication_results["status"] != "ok"]
mae_failures.to_csv(MAE_DIR / "failed_fits.csv", index=False)                                     ### INFRASTRUCTURE
print(f"Completed {expected_count:,} fits in {(perf_counter() - mae_run_start) / 60:.1f} minutes") ### REPORTING
print(f"Failed fits: {len(mae_failures)}; finite large errors are retained.")                      ### REPORTING
if len(mae_failures):                                                                              ### REPORTING
    display(mae_failures[["replication", "T", "method", "error_message"]])                         ### REPORTING

In [ ]:
mae_columns = pd.MultiIndex.from_product([mae_T_values, MAE_METHODS], names=["T", "method"])
mae_loss_matrix = mae_replication_results.pivot(
    index="replication", columns=["T", "method"], values="coefficient_mae"
).reindex(index=range(1, mae_replication_count + 1), columns=mae_columns).to_numpy()
bootstrap_generator = np.random.default_rng(mae_bootstrap_seed)
mae_bootstrap_indices = bootstrap_generator.integers(
    0, mae_replication_count, size=(mae_bootstrap_count, mae_replication_count)
)
# Ordinary means preserve failed fits (NaN); no success-only score is substituted.
mae_bootstrap_values = np.mean(mae_loss_matrix[mae_bootstrap_indices], axis=1)
mae_bootstrap_lower, mae_bootstrap_upper = np.quantile(mae_bootstrap_values, [.025, .975], axis=0)
summary_rows = []
for column, (T, method) in enumerate(mae_columns):
    group = mae_replication_results.loc[
        (mae_replication_results["T"] == T) & (mae_replication_results["method"] == method)
    ]
    failed_count = int((group["status"] != "ok").sum())
    losses = group["coefficient_mae"].to_numpy()
    row = {
        "T": int(T), "method": method, "replications": len(group), "failed_fits": failed_count,
        "mae": float(np.mean(losses)),
        "mae_mc_se": float(np.std(mae_bootstrap_values[:, column], ddof=1)),
        "mae_ci_lower": float(mae_bootstrap_lower[column]),
        "mae_ci_upper": float(mae_bootstrap_upper[column]),
        "median_coefficient_mae": float(np.median(losses)),
        "q90_coefficient_mae": float(np.quantile(losses, .9)),
        "large_mae_count_above_1": int(np.count_nonzero(losses > 1)),
        "nonstationary_estimates": int((group["estimated_stationary"] == False).sum()),
        "mean_runtime_seconds": float(group["runtime_seconds"].mean()),
    }
    for j in range(len(mae_phi)):
        row[f"phi_{j + 1}_mae"] = float(np.mean(group[f"absolute_error_phi_{j + 1}"].to_numpy()))
    summary_rows.append(row)
mae_summary = pd.DataFrame(summary_rows)
mae_summary.to_csv(MAE_DIR / "mae_summary.csv", index=False)
paired_rows = []
for T in mae_T_values:
    gd_column = mae_columns.get_loc((T, "Gradient descent"))
    for comparator in (method for method in MAE_METHODS if method != "Gradient descent"):
        comparator_column = mae_columns.get_loc((T, comparator))
        differences = mae_loss_matrix[:, gd_column] - mae_loss_matrix[:, comparator_column]
        bootstrap_differences = mae_bootstrap_values[:, gd_column] - mae_bootstrap_values[:, comparator_column]
        all_valid = bool(np.all(np.isfinite(differences)))
        paired_rows.append({
            "T": int(T), "comparator": comparator,
            "gradient_descent_minus_comparator_mae": float(np.mean(differences)),
            "mc_se_mae_difference": float(np.std(differences, ddof=1) / np.sqrt(len(differences))),
            "difference_ci_lower": float(np.quantile(bootstrap_differences, .025)),
            "difference_ci_upper": float(np.quantile(bootstrap_differences, .975)),
            "gd_lower_error_replications": int(np.count_nonzero(differences < 0)) if all_valid else np.nan,
            "valid_pairs": int(np.count_nonzero(np.isfinite(differences))),
        })
mae_paired_comparison = pd.DataFrame(paired_rows)
mae_paired_comparison.to_csv(MAE_DIR / "paired_mae_comparisons.csv", index=False)
print("Coefficient MAE (all replications):")
display(mae_summary.pivot(index="T", columns="method", values="mae").round(6))
print("Monte Carlo uncertainty and MAE diagnostics at the largest T:")
display(mae_summary.loc[mae_summary["T"] == mae_T_values[-1], [
    "method", "mae", "mae_ci_lower", "mae_ci_upper", "failed_fits",
    "median_coefficient_mae", "q90_coefficient_mae", "nonstationary_estimates",
]].round(6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.8), constrained_layout=True)                        ### PLOTTING
mae_colors = {"Gradient descent": "tab:blue", "Yule-Walker": "tab:orange",                         ### PLOTTING
               "Conditional QML": "tab:green", "Horvath L-infinity LP": "tab:red"}                  ### PLOTTING
mae_markers = {                                                                                    ### PLOTTING
    "Gradient descent": "o", "Yule-Walker": "s",                                                    ### PLOTTING
    "Conditional QML": "^", "Horvath L-infinity LP": "D",                                           ### PLOTTING
}                                                                                                   ### PLOTTING

for method in MAE_METHODS:                                                                         ### PLOTTING
    values = mae_summary.loc[mae_summary["method"] == method].sort_values("T")                    ### PLOTTING
    valid = np.isfinite(values["mae"]) & (values["mae"] > 0)                                      ### PLOTTING
    if not valid.all():                                                                             ### REPORTING
        print(                                                                                      ### REPORTING
            f"{method}: {int((~valid).sum())} unavailable or zero MAE "                            ### REPORTING
            "values omitted from log-log view."                                                     ### REPORTING
        )                                                                                           ### REPORTING
    for panel, ax in enumerate(axes):                                                               ### PLOTTING
        plotted = values if panel == 0 else values.loc[valid]                                       ### PLOTTING
        ax.plot(                                                                                    ### PLOTTING
            plotted["T"], plotted["mae"], color=mae_colors[method],                               ### PLOTTING
            marker=mae_markers[method], linewidth=1.8, markersize=5,                               ### PLOTTING
            linestyle={"Conditional QML": "--", "Horvath L-infinity LP": "-."}.get(method, "-"),    ### PLOTTING
            label=r"Horváth $L^\infty$ LP" if method == "Horvath L-infinity LP" else method,        ### PLOTTING
        )                                                                                           ### PLOTTING
        ax.fill_between(                                                                            ### PLOTTING
            plotted["T"].to_numpy(), plotted["mae_ci_lower"].to_numpy(),                           ### PLOTTING
            plotted["mae_ci_upper"].to_numpy(), color=mae_colors[method], alpha=0.14,             ### PLOTTING
        )                                                                                           ### PLOTTING

first_mae = mae_summary.loc[mae_summary["T"] == mae_T_values[0], "mae"].to_numpy()             ### PLOTTING
first_mae = first_mae[np.isfinite(first_mae) & (first_mae > 0)]                                 ### PLOTTING
if len(first_mae):                                                                                 ### PLOTTING
    guide_T = np.geomspace(mae_T_values[0], mae_T_values[-1], 200)                                ### PLOTTING
    guide_scale = float(np.median(first_mae))                                                      ### PLOTTING
    for exponent, linestyle, label in (                                                             ### PLOTTING
        (0.5, ":", r"$T^{-1/2}$ comparison"), (1.0, "--", r"$T^{-1}$ comparison")                   ### PLOTTING
    ):                                                                                              ### PLOTTING
        axes[1].plot(                                                                               ### PLOTTING
            guide_T, guide_scale * (guide_T / guide_T[0]) ** (-exponent),                           ### PLOTTING
            color="0.5", linestyle=linestyle, linewidth=1.2, label=label,                           ### PLOTTING
        )                                                                                           ### PLOTTING
for ax in axes:                                                                                     ### PLOTTING
    ax.set_xlabel(r"Sample size $T$")                                                               ### PLOTTING
    ax.set_ylabel(r"Coefficient MAE $(Rp)^{-1}\sum_{r,j}|\widehat{\phi}^{(r)}_{j,T}-\phi_j|$")     ### PLOTTING
    ax.grid(which="both", alpha=0.2)                                                                ### PLOTTING
    ax.legend(fontsize=9, loc="best")                                                               ### PLOTTING
axes[0].set_title("Coefficient MAE versus sample size")                                            ### PLOTTING
axes[0].set_ylim(bottom=0)                                                                          ### PLOTTING
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)                                  ### PLOTTING
axes[1].set_title("Log-log view of convergence speed")                                              ### PLOTTING
axes[1].set_xscale("log")                                                                           ### PLOTTING
axes[1].set_yscale("log")                                                                           ### PLOTTING
fig.suptitle(                                                                                       ### PLOTTING
    fr"AR(3): paired MAE over {mae_replication_count} independent paths "                         ### PLOTTING
    r"(95% Monte Carlo bands)"                                                                      ### PLOTTING
)                                                                                                   ### PLOTTING
mae_figure_stem = "ar3_repeated_coefficient_mae_vs_T"                                             ### PLOTTING
fig.savefig(PDF_DIR / f"{mae_figure_stem}.pdf", bbox_inches="tight")                               ### PLOTTING
fig.savefig(IMAGE_DIR / f"{mae_figure_stem}.png", dpi=180, bbox_inches="tight")                    ### PLOTTING
plt.show()                                                                                          ### PLOTTING
plt.close(fig)                                                                                      ### PLOTTING

print("Paired mean absolute error comparisons at the largest T (negative favors gradient descent):") ### REPORTING
display(mae_paired_comparison.loc[mae_paired_comparison["T"] == mae_T_values[-1]].round(8))      ### REPORTING
print(f"PDF: {PDF_DIR / (mae_figure_stem + '.pdf')}")                                              ### REPORTING
print(f"Detailed results: {MAE_DIR / 'replication_results.csv'}")                                  ### REPORTING